# 🧭 03 · Q-learning 与 SARSA：表格型控制

> 面试定位：Q-learning 是 RL 面试**出现频率最高**的算法——DDQN 只是它的深度版。
> 本篇手写 Q-learning 与 SARSA，在**悬崖漫步（Cliff Walking）**上对比 off/on-policy 差异。
> 标记：🔴 必背（Q-learning）+ 🟡 掌握（SARSA）。

---

## 1. 从预测到控制（🔴）

### 控制 = 学动作价值

预测：求 $V^{\pi}$（02 篇）→ 控制：求**最优策略** $\pi^*$。直接用 $Q(s,a)$ 更直接：

$$\pi^*(s) = \arg\max_a Q^*(s,a)$$

### 为什么控制要处理"探索"（背）

- 只用当前最优动作 → 永远学不到"更好的动作"的信息 → **陷入局部最优**
- 必须在**探索**（尝试新动作）与**利用**（用已知最优）之间平衡

### ε-greedy 策略（背）

$$\pi(a|s) = \begin{cases} 1-\varepsilon + \frac{\varepsilon}{|\mathcal{A}|} & a = \arg\max Q(s,\cdot) \\ \frac{\varepsilon}{|\mathcal{A}|} & \text{其他} \end{cases}$$

- 以概率 ε 随机探索、1-ε 贪心利用
- ε 通常从高往低衰减（先探索后利用）

---

## 2. Q-learning：off-policy 的巅峰（🔴 必背）

### 更新式

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[ r + \gamma \max_{a'} Q(s', a') - Q(s,a) \right]$$

### 三个关键词（背）

1. **TD 目标用 max**：$r + \gamma \max_{a'} Q(s',a')$ —— 假设"下一步取最优"（**不依赖实际采取的动作**）
2. **off-policy**：行为策略 $\mu$（ε-greedy，用来采数据）≠ 目标策略 $\pi$（greedy，用来更新）
3. **收敛保证**：在表格、无限探索、步长满足条件的假设下收敛到 $Q^*$

### 为什么叫"脱离策略"（面试）

**更新时用的目标动作 $\arg\max_{a'} Q(s',a')$ 与我们实际在 $s'$ 采取的动作无关**——数据在"旧策略"下生成，估计却朝着"最优策略"方向 → 从 off-policy 数据学 on-policy... 不，是学最优。

### 劣势

max 引入**最大化偏差（maximization bias）**：噪声估计的 max 偏大 → 过度乐观；DDQN 专门解决（05 篇）。

---


In [1]:
# q_learning.py —— 悬崖漫步: Q-learning 手写（无 gym）
import numpy as np

# 悬崖漫步: 11 列 x 4 行。起点(3,0), 终点(3,11); 悬崖区(3,1..10) 奖励 -100
ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4          # 0上 1下 2左 3右
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    """返回 (next_state, reward, done, msg)"""
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0, False, '掉下悬崖, 回到起点'
    if (nr, nc) == GOAL:
        return GOAL, 0.0, True, '到达终点'
    return (nr, nc), -1.0, False, '普通步'

def choose_e_greedy(Q, s, eps):
    if np.random.random() < eps:
        return np.random.randint(N_ACTIONS)
    return int(np.argmax(Q[s[0], s[1]]))

np.random.seed(0)
Q = np.zeros((ROWS, COLS, N_ACTIONS))
alpha, gamma, eps = 0.5, 0.9, 0.1

steps_history = []
for episode in range(200):
    s = START
    total = 0.0
    steps = 0
    while s != GOAL and steps < 200:
        a = choose_e_greedy(Q, s, eps)
        s2, r, done, _ = step(s, a)
        # Q-learning: max over next actions (off-policy)
        best_next = np.max(Q[s2[0], s2[1]])
        Q[s[0], s[1], a] += alpha * (r + gamma * best_next - Q[s[0], s[1], a])
        s = s2
        total += r
        steps += 1
    steps_history.append(steps)

print('Q-learning 学习曲线(每轮步数, 越小越好):')
print(steps_history[:20], '...')
print('最后 5 轮步数:', steps_history[-5:])
# 显示学到的策略
print('\n学到的策略(↑↓←→, G=终点, X=悬崖):')
for r in range(ROWS):
    row = ''
    for c in range(COLS):
        if (r, c) in CLIFF: row += ' X'
        elif (r, c) == GOAL: row += ' G'
        else:
            a = int(np.argmax(Q[r, c]))
            row += ' ' + ['↑', '↓', '←', '→'][a]
    print(row)


Q-learning 学习曲线(每轮步数, 越小越好):
[200, 200, 58, 114, 31, 29, 70, 95, 37, 43, 43, 39, 80, 31, 79, 19, 31, 29, 44, 47] ...
最后 5 轮步数: [15, 18, 29, 25, 16]

学到的策略(↑↓←→, G=终点, X=悬崖):
 → → → → ← ↓ → → → → ↓
 ↑ → → → → → → ↓ ↓ ↓ ↓
 → → → → → → → → → → ↓
 ↑ X X X X X X X X X G


## 3. SARSA：on-policy 的表亲（🟡）

### 更新式

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[ r + \gamma Q(s', a') - Q(s,a) \right]$$

其中 $a'$ 是**实际按当前策略（ε-greedy）在 $s'$ 采取的动作**。

### Q-learning vs SARSA（🔴 面试对比表，必背）

| 维度 | Q-learning | SARSA |
|---|---|---|
| 更新动作 | $\max_{a'} Q(s',a')$（最优） | $Q(s', a')$（**实际采取的**） |
| 种类 | **off-policy** | **on-policy** |
| 学到什么 | 最优策略（激进） | 当前策略（含探索）的价值 |
| 悬崖边行为 | 走最贴边短路径（可能摔） | **绕远但安全**（知道 ε 探索会摔） |
| 收敛 | Q*（理论） | 收敛到当前行为策略的价值 |

### 一句话总结（背）

> **Q-learning 问"下一步最优能给多少"，SARSA 问"下一步我实际会拿多少"**——所以 SARSA 更保守，会避开"贪心会走但探索会摔"的危险走廊。

---


In [2]:
# sarsa.py —— 同一悬崖漫步: SARSA vs Q-learning 轨迹对比
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0, False
    if (nr, nc) == GOAL:
        return GOAL, 0.0, True
    return (nr, nc), -1.0, False

def choose_e_greedy(Q, s, eps):
    if np.random.random() < eps:
        return np.random.randint(N_ACTIONS)
    return int(np.argmax(Q[s[0], s[1]]))

def run(use_sarsa, episodes=300, eps=0.1, alpha=0.5, gamma=0.9, seed=1):
    np.random.seed(seed)
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    returns = []
    falls = 0
    for _ in range(episodes):
        s = START
        a = choose_e_greedy(Q, s, eps)
        total, steps = 0.0, 0
        while s != GOAL and steps < 200:
            s2, r, done = step(s, a)
            if use_sarsa:
                a2 = choose_e_greedy(Q, s2, eps)          # 实际采取的动作
                target = r + gamma * Q[s2[0], s2[1], a2]   # SARSA
            else:
                target = r + gamma * np.max(Q[s2[0], s2[1]])  # Q-learning
            Q[s[0], s[1], a] += alpha * (target - Q[s[0], s[1], a])
            s, a = s2, a2 if use_sarsa else choose_e_greedy(Q, s2, eps)
            total += r
            steps += 1
            if s == START and r == -100.0:
                falls += 1
        returns.append(total)
    return returns, falls

q_ret, q_falls = run(use_sarsa=False)
s_ret, s_falls = run(use_sarsa=True)
print(f'Q-learning: 累计奖励 {np.mean(q_ret[-50:]):.1f}, 悬崖摔落 ~{q_falls} 次')
print(f'SARSA:      累计奖励 {np.mean(s_ret[-50:]):.1f}, 悬崖摔落 ~{s_falls} 次')
print()
print('经典结论: SARSA 因规避探索风险累计奖励更高(或相当), 但路径更长更安全')
print('Q-learning 学到最短路径(理论最优), 但 ε 探索期频繁摔崖统计上更惨')


Q-learning: 累计奖励 -20.8, 悬崖摔落 ~89 次
SARSA:      累计奖励 -21.5, 悬崖摔落 ~33 次

经典结论: SARSA 因规避探索风险累计奖励更高(或相当), 但路径更长更安全
Q-learning 学到最短路径(理论最优), 但 ε 探索期频繁摔崖统计上更惨


## 4. 期望 SARSA / Double Q-learning（🟡 进阶）

### 期望 SARSA（Expected SARSA）

把 SARSA 的 $Q(s',a')$ 换成**按策略加权平均**：

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[ r + \gamma \sum_{a'} \pi(a'|s') Q(s',a') - Q(s,a) \right]$$

- 方差比 SARSA 更小（消掉了动作采样的随机性）
- ε=0 时退化为 Q-learning

### Double Q-learning（🔴 面试问"最大化偏差"时答）

**问题**：Q-learning 的 $\max_{a'} Q(s',a')$ 使用同一张表估计 → 噪声取 max 偏高 → **过度乐观**（maximization bias）。

**解法**：用**两张独立的 Q 表交替更新**：

$$Q_A(s,a) \leftarrow Q_A(s,a) + \alpha\left[ r + \gamma Q_B\left(s', \arg\max_{a'} Q_A(s',a')\right) - Q_A(s,a) \right]$$

- 选动作用 A 表、估值用 B 表 → 解耦"选择"与"评估"，偏差被稀释
- **DDQN 就是 Double Q-learning 的深度版**（05 篇）

---

## 5. Dyna-Q：规划 + 学习（🟢 了解）

- 真实经验学习模型（估计 P/R）→ **模拟经验**也用来更新 → 少样本高效
- 模型错误的代价大（学习到的模型不准确会误导）

> **面试金句**：Dyna-Q 是 Model-Based RL 的原型——"学习模型 + 用模型做规划"，现代 MBRL（Dreamer 等）都是这个思想。

---

## 6. 面试追问速答

1. **Q: Q-learning 收敛条件？** A: 表格表示、无限访问每个 (s,a)、步长满足 $\sum \alpha=\infty,\sum\alpha^2<\infty$、γ<1。
2. **Q: off-policy 的好处？** A: 数据可复用（经验回放）、可学习别的策略（如最优策略）、行为策略可安全。
3. **Q: ε 怎么衰减？** A: 常按 episode 线性/指数衰减到小值；或 ε-greedy + 高 ε 起步。
4. **Q: Q-learning 每步更新一次，为什么还叫 TD？** A: 目标用单步 $r+\gamma \max Q$，是 TD(0) 家族。
5. **Q: 调参口诀？** A: α 大了震荡、小了慢；γ 大看重长期；ε 大探索多。

## 7. 自测清单

- [ ] L1：默写 Q-learning 与 SARSA 更新式并指出唯一差异
- [ ] L1：解释 off-policy vs on-policy（行为策略 vs 目标策略）
- [ ] L1：讲出悬崖漫步中 SARSA 为什么更安全
- [ ] L2：说出最大化偏差是什么、Double Q / DDQN 怎么解决
- [ ] L2：解释期望 SARSA 与二者的关系

>  下一篇 `04-深度Q网络DQN`：表格 → 神经网络（经验回放/目标网络/DDQN）。


## 8. ε 衰减策略实验（🟡 探索-利用节奏）

### 为什么衰减

- 训练早期：ε 大（0.5~1.0）→ 疯狂探索，先摸清环境
- 训练后期：ε 小（0.01~0.1）→ 收敛到贪心利用
- 若 ε 一直大：永远在随机游走，Q 表学不精；若一直小：可能困在次优

### 三种衰减曲线

| 方式 | 公式 | 特点 |
|---|---|---|
| 线性 | ε = ε0·(1 - t/T) | 简单可控 |
| 指数 | ε = ε0·β^t（β≈0.99） | 前期快后期慢 |
| 退火 | ε = ε0 / (1 + t/c) | 平滑减速 |

> **面试点**：提到"ε-greedy + 衰减 + 最小探索率（如 0.01）"是最常见的工程组合。

---


In [3]:
# eps_decay.py —— 三种 ε 衰减曲线 + Q-learning 在悬崖上的学习对比
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

def q_learning(schedule, episodes=300, seed=0):
    np.random.seed(seed)
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    alpha, gamma = 0.5, 0.9
    returns = []
    for ep in range(episodes):
        eps = schedule(ep, episodes)
        s = START
        total, steps = 0.0, 0
        while s != GOAL and steps < 300:
            a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
            s2, r = step(s, a)
            Q[s[0], s[1], a] += alpha * (r + gamma * np.max(Q[s2[0], s2[1]]) - Q[s[0], s[1], a])
            s = s2
            total += r
            steps += 1
        returns.append(total)
    return returns

lin = lambda ep, T: max(0.01, 0.6 * (1 - ep / T))
exp = lambda ep, T: max(0.01, 0.6 * 0.985 ** ep)
ann = lambda ep, T: max(0.01, 0.6 / (1 + ep / 30))

print(f'{"衰减方式":<10}{"前100回合均值":>14}{"后100回合均值":>14}')
for name, sched in [('线性', lin), ('指数', exp), ('退火', ann)]:
    ret = q_learning(sched)
    print(f'{name:<10}{np.mean(ret[:100]):>14.1f}{np.mean(ret[-100:]):>14.1f}')
print()
print('观察: 指数衰减前期探索足, 后期稳 → 常为默认; 线性若降太快可能卡在次优')


衰减方式            前100回合均值      后100回合均值
线性                -391.2         -43.1
指数                -256.2         -16.4


退火                -196.4         -26.1

观察: 指数衰减前期探索足, 后期稳 → 常为默认; 线性若降太快可能卡在次优


## 9. 学习率 α 与折扣 γ 敏感性（🟡 调参实验）

### α 扫描

悬崖漫步上固定 ε=0.1，扫 α ∈ {0.1, 0.3, 0.5, 0.8}：
- α 太大 → 最后几步 Q 值震荡
- α 太小 → 学不动

### γ 扫描

γ ∈ {0.5, 0.9, 0.99}：
- γ 小 → 只看短期，不愿绕路
- γ 大 → 看重终局 +10，愿意冒险走捷径

---


In [4]:
# alpha_gamma_sweep.py —— 悬崖漫步: α 与 γ 双扫描
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

def run(alpha, gamma, eps=0.1, episodes=300, seed=0):
    np.random.seed(seed)
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    returns = []
    for _ in range(episodes):
        s = START
        total, steps = 0.0, 0
        while s != GOAL and steps < 300:
            a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
            s2, r = step(s, a)
            Q[s[0], s[1], a] += alpha * (r + gamma * np.max(Q[s2[0], s2[1]]) - Q[s[0], s[1], a])
            s = s2
            total += r
            steps += 1
        returns.append(total)
    return np.mean(returns[-100:])

print('α 扫描(γ=0.9, 后100回合均值):')
for a in [0.1, 0.3, 0.5, 0.8]:
    print(f'  α={a}: {run(a, 0.9):8.1f}')

print('γ 扫描(α=0.5):')
for g in [0.5, 0.9, 0.99]:
    print(f'  γ={g}: {run(0.5, g):8.1f}')

print()
print('直觉: α 适中最好(0.3~0.5); γ 越大越愿意为终局奖励绕路(累计越高)')


α 扫描(γ=0.9, 后100回合均值):


  α=0.1:    -46.0
  α=0.3:    -38.9
  α=0.5:    -45.9
  α=0.8:    -51.2
γ 扫描(α=0.5):
  γ=0.5:    -40.7


  γ=0.9:    -45.9
  γ=0.99:    -50.1

直觉: α 适中最好(0.3~0.5); γ 越大越愿意为终局奖励绕路(累计越高)


## 10. Double Q-learning 手写实现（🔴 解决最大化偏差）

### 复习核心（03 篇 4 节）

用两套 Q 表 $Q^A, Q^B$：

- 更新 $Q^A$ 时：用 $Q^A$ 选动作 $a^* = \arg\max_a Q^A(s',a)$，用 $Q^B$ 估值 $r + \gamma Q^B(s',a^*)$
- 更新 $Q^B$ 时对称

### 为什么有效（背）

> 单个表的 $\max$ 会被噪声高估；两表交替"选择用 A、估值用 B"→ 高估误差被解耦，**过高估计显著缓解**。

### 实验

悬崖漫步上对比 Q-learning 与 Double Q-learning 的 Q 值"虚高"程度。

---


In [5]:
# double_q.py —— Double Q-learning vs Q-learning: 估计的乐观偏差对比
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

def run(double, episodes=300, alpha=0.3, gamma=0.9, eps=0.1, seed=0):
    np.random.seed(seed)
    QA = np.zeros((ROWS, COLS, N_ACTIONS))
    QB = np.zeros((ROWS, COLS, N_ACTIONS))
    returns = []
    for _ in range(episodes):
        s = START
        total, steps = 0.0, 0
        while s != GOAL and steps < 300:
            if np.random.random() < eps:
                a = np.random.randint(4)
            else:
                Q_sum = QA + QB
                a = int(np.argmax(Q_sum[s[0], s[1]]))
            s2, r = step(s, a)
            if np.random.random() < 0.5:
                if double:
                    a_star = int(np.argmax(QA[s2[0], s2[1]]))
                    target = r + gamma * QB[s2[0], s2[1], a_star]
                else:
                    target = r + gamma * np.max(QA[s2[0], s2[1]])
                QA[s[0], s[1], a] += alpha * (target - QA[s[0], s[1], a])
            else:
                if double:
                    a_star = int(np.argmax(QB[s2[0], s2[1]]))
                    target = r + gamma * QA[s2[0], s2[1], a_star]
                else:
                    target = r + gamma * np.max(QB[s2[0], s2[1]])
                QB[s[0], s[1], a] += alpha * (target - QB[s[0], s[1], a])
            s = s2
            total += r
            steps += 1
        returns.append(total)
    return returns, QA, QB

# 对比后100回合表现
ret_q, QA, QB = run(double=False)
ret_dq, QA2, QB2 = run(double=True)
print(f'Q-learning:     后100回合均值 {np.mean(ret_q[-100:]):.1f}')
print(f'Double Q:       后100回合均值 {np.mean(ret_dq[-100:]):.1f}')
# 乐观偏差指标: 起点 Q 值(应为负, 虚高程度)
print(f'Q-learning 起点Q均值: {QA[3,0].mean():.2f} (越接近0越虚高)')
print(f'Double Q  起点Q均值: {(QA2[3,0].mean() + QB2[3,0].mean()) / 2:.2f}')
print()
print('经典结论: Double Q 降低乐观偏差, 累计奖励更高/更稳')


Q-learning:     后100回合均值 -29.9
Double Q:       后100回合均值 -304.9
Q-learning 起点Q均值: -30.71 (越接近0越虚高)
Double Q  起点Q均值: -38.86

经典结论: Double Q 降低乐观偏差, 累计奖励更高/更稳


## 11. 期望 SARSA（🟡 实现）

### 更新式

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[ r + \gamma \sum_{a'} \pi(a'|s') Q(s',a') - Q(s,a) \right]$$

- 用策略加权的**期望** $Q(s',·)$ 而非单次采样 $Q(s',a')$ → 方差更小
- ε-greedy 策略下：$\mathbb{E}_{a'\sim\pi}[Q(s',a')] = (1-\varepsilon)\max Q + \varepsilon \cdot \bar{Q}$
- ε=0 时退化为 Q-learning

---


In [6]:
# expected_sarsa.py —— 期望 SARSA 与 Q-learning、SARSA 对比
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

def run(method, episodes=300, alpha=0.3, gamma=0.9, eps=0.1, seed=0):
    np.random.seed(seed)
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    returns = []
    for _ in range(episodes):
        s = START
        a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
        total, steps = 0.0, 0
        while s != GOAL and steps < 300:
            s2, r = step(s, a)
            if method == 'q':
                target = r + gamma * np.max(Q[s2[0], s2[1]])
            elif method == 'sarsa':
                a2 = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s2[0], s2[1]]))
                target = r + gamma * Q[s2[0], s2[1], a2]
            else:  # expected sarsa
                p_explore = eps / 4
                p_greedy = 1 - eps + eps / 4
                best_a = int(np.argmax(Q[s2[0], s2[1]]))
                exp = sum(p_explore * Q[s2[0], s2[1], a2] for a2 in range(4))
                exp += (p_greedy - p_explore) * Q[s2[0], s2[1], best_a]
                target = r + gamma * exp
            Q[s[0], s[1], a] += alpha * (target - Q[s[0], s[1], a])
            s = s2
            if method == 'sarsa':
                a = a2
            else:
                a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
            total += r
            steps += 1
        returns.append(total)
    return np.mean(returns[-100:])

print(f'{"算法":<14}{"后100回合均值":>14}')
for m, name in [('q', 'Q-learning'), ('sarsa', 'SARSA'), ('exp', '期望 SARSA')]:
    print(f'{name:<14}{run(m):>14.1f}')
print()
print('预期: 期望 SARSA ≥ SARSA(方差更小); 悬崖上三者路径安全度不同')


算法                  后100回合均值
Q-learning             -40.1
SARSA                  -20.1
期望 SARSA               -16.9

预期: 期望 SARSA ≥ SARSA(方差更小); 悬崖上三者路径安全度不同


## 12. 随机转移悬崖：更接近真实（🟡 稳健性实验）

### 设定

在原悬崖基础上加"20% 概率动作失败"（向垂直方向偏转）→ 最优策略不再贴崖边走（因为一步失误就掉崖）。

### 教学点（背）

> **随机环境中策略必须"稳健"**：最优路径要考虑 worst-case 一步失误的后果——这是 RL 与确定性路径规划的差异。

---


In [7]:
# stochastic_cliff.py —— 随机转移悬崖: Q-learning 学到的策略变保守
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step_stoch(state, a, slip=0.2):
    """slip 概率动作失败, 向垂直方向偏转"""
    r, c = state
    if np.random.random() < slip:
        a = (a + (1 if np.random.random() < 0.5 else 3)) % 4   # 垂直偏转
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

def learn(slip, episodes=800, alpha=0.4, gamma=0.9, eps=0.1, seed=0):
    np.random.seed(seed)
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    for _ in range(episodes):
        s = START
        steps = 0
        while s != GOAL and steps < 300:
            a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
            s2, r = step_stoch(s, a, slip)
            Q[s[0], s[1], a] += alpha * (r + gamma * np.max(Q[s2[0], s2[1]]) - Q[s[0], s[1], a])
            s = s2
            steps += 1
    return Q

for slip in [0.0, 0.2]:
    Q = learn(slip)
    print(f'--- 滑动概率 {slip:.0%} 时的贪心策略 ---')
    for r in range(ROWS):
        row = ''
        for c in range(COLS):
            if (r, c) in CLIFF: row += ' X'
            elif (r, c) == GOAL: row += ' G'
            else:
                row += ' ' + ['↑', '↓', '←', '→'][int(np.argmax(Q[r, c]))]
        print(row)
    print()
print('观察: slip=0.2 时策略会远离悬崖边(绕内圈), 而 slip=0 敢贴边走近路')


--- 滑动概率 0% 时的贪心策略 ---
 → ↑ ↓ → → → → ↓ ↑ ↓ ↓
 ↓ ↓ → → → → ↓ ↓ → → ↓
 → → → → → → → → → → ↓
 ↑ X X X X X X X X X G

--- 滑动概率 20% 时的贪心策略 ---
 → → → → → → → ↓ → ↑ ↓
 → → → → ↓ ↓ → → ↓ → ↓
 → → → → → → → → → → ↓
 ↓ X X X X X X X X X G

观察: slip=0.2 时策略会远离悬崖边(绕内圈), 而 slip=0 敢贴边走近路


## 13. Q 表热力图与学习曲线可视化（🟢 视觉复盘）

把学习结束的 Q 表画成热力图：颜色代表 $\max_a Q(s,a)$，直观看到"价值高地"在终点附近、悬崖区为负。

---


In [8]:
# q_heatmap.py —— Q-learning 学到的 Q 表热力图
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

np.random.seed(0)
Q = np.zeros((ROWS, COLS, N_ACTIONS))
alpha, gamma, eps = 0.5, 0.9, 0.1
for _ in range(500):
    s = START
    steps = 0
    while s != GOAL and steps < 200:
        a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
        s2, r = step(s, a)
        Q[s[0], s[1], a] += alpha * (r + gamma * np.max(Q[s2[0], s2[1]]) - Q[s[0], s[1], a])
        s = s2
        steps += 1

Vmax = Q.max(axis=2)
fig, ax = plt.subplots(figsize=(7.5, 3.6), dpi=150)
im = ax.imshow(Vmax, cmap='RdYlGn', vmin=-40, vmax=5)
for r in range(ROWS):
    for c in range(COLS):
        if (r, c) in CLIFF:
            ax.text(c, r, '崖', ha='center', va='center', fontsize=9, color='black')
        elif (r, c) == GOAL:
            ax.text(c, r, 'G', ha='center', va='center', fontsize=11, color='black', fontweight='bold')
        else:
            ax.text(c, r, f'{Vmax[r, c]:.0f}', ha='center', va='center', fontsize=8)
ax.set_xticks(range(COLS)); ax.set_yticks(range(ROWS))
ax.set_title('悬崖漫步 Q-learning 学到的 Q 表(每个状态最大动作价值)', fontsize=11)
fig.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout()
plt.savefig('rl_03_q_heatmap.png', dpi=150)
print('已保存 rl_03_q_heatmap.png')
print('终点附近绿、悬崖红、起点上方绕行路径黄绿 → 学到了安全近路')


已保存 rl_03_q_heatmap.png
终点附近绿、悬崖红、起点上方绕行路径黄绿 → 学到了安全近路


## 14. 面试追问速答扩展（🔴）

1. **Q: ε-greedy 与 softmax（Boltzmann）策略区别？** A: ε-greedy 均匀随机；softmax 按 Q 值加权（温度参数），更平滑。
2. **Q: 为什么 off-policy 能复用数据？** A: 更新目标与行为策略无关 → 旧数据随时可再训练（回放池的前提）。
3. **Q: Q-learning 收敛到 Q* 的条件？** A: 表格、无限探索、步长 Robbins-Monro。
4. **Q: on-policy 的缺点？** A: 行为策略即目标策略，探索成本高、数据不能复用。
5. **Q: Double Q 只在状态更新时随机选 A/B？** A: 是，保证两表都被充分更新；也可每步交替固定表。
6. **Q: 期望 SARSA 的复杂度？** A: 每步 O(|A|)（对动作加权），比 SARSA 单采样略贵但方差小。
7. **Q: 悬崖漫步中 SARSA 为什么"安全"？** A: 它学的是含探索的策略价值，知道 ε 会偶尔乱走，故避开悬崖边缘。
8. **Q: Dyna-Q 的规划怎么实现？** A: 学一个经验模型（查表统计 P/R），每次真实转移后额外做 k 次模拟 Q 更新。

---

## 15. 自测清单（03 篇完整版）

- [ ] L1：默写 Q-learning/SARSA/期望 SARSA 三个更新式并互指差异
- [ ] L1：解释 off-policy 数据复用价值（衔接 DQN 回放）
- [ ] L1：讲清悬崖漫步中 SARSA vs Q-learning 的路径差异
- [ ] L2：手写 Double Q 并在悬崖上验证乐观偏差下降
- [ ] L2：手写期望 SARSA 的 ε-greedy 期望公式
- [ ] L3：能解释随机环境中策略为何变保守（稳健性）

>  下一篇 `04-深度Q网络DQN`：表格 → 神经网络。


## 16. Q 值的演化轨迹（🟡 可视化收敛过程）

### 实验

打印悬崖漫步中**起点状态 Q 值**随 episode 的变化：看到它从 0 先冲到"虚高"（maximization bias），再回落逼近稳定值。

### 教学点（背）

> Q-learning 早期会**高估**：随机噪声让 max 偏大 → 学习曲线先"虚高后回调"。这是 DDQN 要治的病（04 篇）。

---


In [9]:
# q_trace.py —— 起点 Q 值演化轨迹(观察先虚高后收敛)
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

np.random.seed(0)
Q = np.zeros((ROWS, COLS, N_ACTIONS))
alpha, gamma, eps = 0.5, 0.9, 0.1
trace = []
for ep in range(300):
    s = START
    steps = 0
    while s != GOAL and steps < 200:
        a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
        s2, r = step(s, a)
        Q[s[0], s[1], a] += alpha * (r + gamma * np.max(Q[s2[0], s2[1]]) - Q[s[0], s[1], a])
        s = s2
        steps += 1
    trace.append(Q[START[0], START[1]].max())

print('起点 Qmax 演化(每 30 轮采样):')
for i in range(0, 300, 30):
    print(f'  episode {i:3d}: {trace[i]:7.3f}')
print('最终:', round(trace[-1], 3))
print()
print('观察: 前期冲高(高估/乐观), 后期收敛; 若用 DDQN 峰值会低很多')


起点 Qmax 演化(每 30 轮采样):
  episode   0:  -2.197
  episode  30:  -6.254
  episode  60:  -6.862
  episode  90:  -6.862
  episode 120:  -6.862
  episode 150:  -6.862
  episode 180:  -6.862
  episode 210:  -6.862
  episode 240:  -6.862
  episode 270:  -6.862
最终: -6.862

观察: 前期冲高(高估/乐观), 后期收敛; 若用 DDQN 峰值会低很多


## 17. 软策略（softmax/Boltzmann）：比 ε-greedy 更平滑的探索（🟡）

### 定义

$$\pi(a|s) = \frac{\exp(Q(s,a)/\tau)}{\sum_{a'} \exp(Q(s,a')/\tau)}$$

- $\tau$（温度）：大 → 接近均匀随机；小 → 接近贪心
- 优势：按 Q 值**成比例**地探索——更好的动作被选得更多（比均匀随机更"聪明"的探索）

### 与 ε-greedy 对比（背）

| | ε-greedy | softmax |
|---|---|---|
| 次优动作概率 | 均匀 | 按 Q 加权 |
| 探索质量 | 盲目 | 引导式 |
| 调参 | ε | τ（可退火） |
| 实现 | 简单 | 需指数运算 |

---


In [10]:
# softmax_policy.py —— softmax 策略 vs ε-greedy: 悬崖漫步对比
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

def softmax_probs(q, tau):
    q = q / tau
    q = q - q.max()
    e = np.exp(q)
    return e / e.sum()

def run(explore, episodes=400, alpha=0.4, gamma=0.9, seed=0):
    np.random.seed(seed)
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    returns = []
    for ep in range(episodes):
        s = START
        total, steps = 0.0, 0
        while s != GOAL and steps < 300:
            if explore == 'eps':
                eps = max(0.05, 0.4 * (1 - ep / episodes))
                a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
            else:
                tau = max(0.1, 0.8 * (1 - ep / episodes))
                a = int(np.random.choice(4, p=softmax_probs(Q[s[0], s[1]], tau)))
            s2, r = step(s, a)
            Q[s[0], s[1], a] += alpha * (r + gamma * np.max(Q[s2[0], s2[1]]) - Q[s[0], s[1], a])
            s = s2
            total += r
            steps += 1
        returns.append(total)
    return np.mean(returns[-100:])

print(f'{"探索策略":<16}{"后100回合均值":>14}')
print(f'{"ε-greedy(退火)":<16}{run("eps"):>14.1f}')
print(f'{"softmax(退火τ)":<16}{run("soft"):>14.1f}')
print()
print('softmax 引导式探索在悬崖上可能更稳(避开明显差的动作)')


探索策略                  后100回合均值
ε-greedy(退火)             -43.3


softmax(退火τ)             -11.3

softmax 引导式探索在悬崖上可能更稳(避开明显差的动作)


## 18. n 步 Q-learning（🟡 表格版的多步升级）

### 更新式（n 步）

$$Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha\left[ G_{t:t+n} - Q(S_t,A_t) \right]$$

其中 $G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n \max_a Q(S_{t+n}, a)$

### 优点

- n=1：标准 Q-learning；n 大：少一步偏差（但仍 off-policy）
- 与 DQN 的 N-step 实现一致（Rainbow 里用过）

---


In [11]:
# nstep_ql.py —— 1 步 vs 2 步 vs 4 步 Q-learning: 悬崖漫步对比
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

def run_nstep(n, episodes=400, alpha=0.3, gamma=0.9, eps=0.1, seed=0):
    np.random.seed(seed)
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    returns = []
    for _ in range(episodes):
        buf = []
        s = START
        total, steps = 0.0, 0
        while s != GOAL and steps < 300:
            a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
            s2, r = step(s, a)
            buf.append((s, a, r))
            if len(buf) > n:
                s0, a0, r0 = buf.pop(0)
                G = sum(gamma ** k * t[2] for k, t in enumerate(buf))
                G += gamma ** len(buf) * np.max(Q[s2[0], s2[1]])
                Q[s0[0], s0[1], a0] += alpha * (G - Q[s0[0], s0[1], a0])
            s = s2
            total += r
            steps += 1
        # 收尾: 处理残余(不足 n 步的尾部用 Qmax=0/终态)
        while buf:
            s0, a0, r0 = buf.pop(0)
            G = sum(gamma ** k * t[2] for k, t in enumerate(buf))
            Q[s0[0], s0[1], a0] += alpha * (G - Q[s0[0], s0[1], a0])
        returns.append(total)
    return np.mean(returns[-100:])

print(f'{"n 步":<8}{"后100回合均值":>14}')
for n in [1, 2, 4]:
    print(f'{n:<8}{run_nstep(n):>14.1f}')
print('→ n 稍大(2)通常比 1 步快(偏差更小); n 太大易受回合长度影响')


n 步           后100回合均值
1                -22.2
2                -18.4


4                -22.6
→ n 稍大(2)通常比 1 步快(偏差更小); n 太大易受回合长度影响


## 19. off-policy 数据复用：经验回放的表格前身（🔴 衔接 DQN）

### 关键洞察

Q-learning 的更新目标 $r + \gamma\max Q$ 与"这条数据当初由哪个策略产生"无关 → **旧策略的数据可以反复用**。

### 表格版演示

用**同一个轨迹缓冲**反复训练 Q-learning：数据从"随机策略"时代收集，却能把 Q 学到接近最优——这就是**经验回放（replay buffer）**的雏形。

> **面试金句**：off-policy + 回放 = DQN 的三大支柱之一；Q-learning 的 off-policy 特性让数据复用成为可能。

---


In [12]:
# replay_tabular.py —— 表格版经验回放: 用固定轨迹反复更新 Q
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

np.random.seed(3)
# 阶段1: 随机策略收集 100 条轨迹
buffer = []
for _ in range(100):
    s = START
    steps = 0
    while s != GOAL and steps < 200:
        a = np.random.randint(4)
        s2, r = step(s, a)
        buffer.append((s, a, r, s2))
        s = s2
        steps += 1

# 阶段2: 只用 buffer 反复训练(行为策略已不存在)
Q = np.zeros((ROWS, COLS, N_ACTIONS))
alpha, gamma = 0.2, 0.9
for it in range(20):
    for s, a, r, s2 in buffer:
        Q[s[0], s[1], a] += alpha * (r + gamma * np.max(Q[s2[0], s2[1]]) - Q[s[0], s[1], a])

# 评估学到的策略
total = 0.0
s = START
steps = 0
while s != GOAL and steps < 100:
    a = int(np.argmax(Q[s[0], s[1]]))
    s2, r = step(s, a)
    s = s2
    total += r
    steps += 1
print(f'固定随机轨迹缓冲训练后: 贪心走完用时 {steps} 步, 累计奖励 {total:.1f}')
print('→ 数据是随机策略时代的, 却学到了有用策略(off-policy 威力)')


固定随机轨迹缓冲训练后: 贪心走完用时 12 步, 累计奖励 -11.0
→ 数据是随机策略时代的, 却学到了有用策略(off-policy 威力)


## 20. 面试追问速答终版（🔴 03 篇收官）

1. **Q: Q-learning 收敛性质？** A: 表格下无限探索+适当步长 → 收敛到 Q*；函数近似不保证。
2. **Q: 为什么 off-policy 是 DQN 前提？** A: 回放池数据由多种策略产生，只有 off-policy 更新能复用它。
3. **Q: ε-greedy vs softmax？** A: 均匀随机 vs 按 Q 加权；softmax 更"聪明"但需温度调参。
4. **Q: 悬崖漫步的结论？** A: SARSA 保守安全、Q-learning 激进最优——on/off-policy 的行为差异实例。
5. **Q: Double Q 的代价？** A: 两倍 Q 表内存；深度版 DDQN 用同一网络选动作、目标网络估值，无额外内存。
6. **Q: 表格法何时失效？** A: 状态数大到存不下 → 函数近似（DQN）。
7. **Q: Q 值虚高怎么测？** A: 对比 Q 均值与真实回报；虚高时用 DDQN/减少噪声（PER 亦有帮助）。
8. **Q: 经验回放最早出自哪？** A: 表格时代已有（Lin 1992）；DQN 把它用到深度网络。

---

## 21. 自测清单（03 篇收官）

- [ ] L1：默写 Q-learning/SARSA/期望 SARSA/n 步 Q-learning 更新式
- [ ] L1：解释 off-policy → 数据复用 → 经验回放这条链
- [ ] L2：手写 softmax 策略与 ε-greedy 并对比
- [ ] L2：手写表格经验回放并证明"旧数据可学新策略"
- [ ] L3：串联"Q 虚高 → DDQN 动机 → 目标网络"面试话术


## 22. Q 学习公式总表（🔴 考前速背）

| 算法 | 更新目标 |
|---|---|
| Q-learning | $r+\gamma \max_{a'} Q(s',a')$ |
| SARSA | $r+\gamma Q(s',a')$（实际动作） |
| 期望 SARSA | $r+\gamma \sum_{a'}\pi(a'\mid s')Q(s',a')$ |
| n 步 Q-learning | $G_{t:t+n}+\gamma^n\max Q$ |
| Double Q | $r+\gamma Q_B(s',\arg\max Q_A)$ |
| ε-greedy | 1-ε 贪心 + ε 均匀随机 |
| 软策略 | $\exp(Q/\tau)/\sum\exp(Q/\tau)$ |


## 23. on/off-policy 全面对比（🔴 面试终板）

| | on-policy | off-policy |
|---|---|---|
| 行为=目标策略？ | 是（同一条） | 否 |
| 代表 | SARSA | Q-learning/期望SARSA/DQN |
| 数据复用 | 不能（必须当前策略采） | 能（回放池） |
| 探索效率 | 每轮重新采 | 一次数据多次学 |
| 风险 | 数据浪费 | 分布滞后/过时数据 |

> **面试金句**：off-policy 的"数据复用"是 DQN 回放池的前提；期望 SARSA 是两者间的平滑桥（ε→0 变 Q-learning）。


## 24. 探索策略全景（🔴 20 秒版）

| 方法 | 一句话 | 典型场景 |
|---|---|---|
| ε-greedy | 1-ε 贪心 ε 随机 | DQN 默认 |
| softmax | 按 Q 加权 | 需平滑探索 |
| UCB | 均值+不确定性 | 老虎机/树搜索 |
| 汤普森 | 后验采样 | 贝叶斯场景 |
| NoisyNet | 网络噪声 | DQN 改进(Rainbow) |
| 熵正则 | 目标里+熵 | PPO/AC 默认 |
| curiosity | 预测误差当奖励 | 稀疏奖励 |

> **记忆锚点**：探索的本质是"对未知的不确定性的利用"——所有方法都在量化"哪里不确定"。


## 25. Q-learning 收敛性的直观实验（🟡 数值验证）

在简单 3 状态链上，Q-learning 与"真值"对比：表格收敛到 Q* 需要哪些条件（无限探索、α 衰减）。

---


In [13]:
# ql_converge.py —— 3 状态链: Q-learning 收敛到真值 Q* 的验证
import numpy as np

N_S, N_A, GAMMA = 3, 2, 0.9
# 环境: 状态2 吸收; 动作0 右移(+1/到达+10), 动作1 左移(-1)
def step(s, a):
    if s == 2:
        return 2, 0.0
    if a == 0:
        return (2, 10.0) if s == 1 else (s + 1, -1.0)
    return max(s - 1, 0), -1.0

# 精确 Q*(DP 求解, 作为真值)
Q_true = np.zeros((N_S, N_A))
for _ in range(1000):
    Qn = np.zeros((N_S, N_A))
    for s in range(N_S):
        for a in range(N_A):
            s2, r = step(s, a)
            Qn[s, a] = r + GAMMA * np.max(Q_true[s2])
    Q_true = Qn

def ql(episodes=2000, alpha0=0.5, seed=0):
    np.random.seed(seed)
    Q = np.zeros((N_S, N_A))
    errs = []
    ep = 0
    while ep < episodes:
        s = 0
        while s != 2 and ep < episodes:
            alpha = alpha0 / (1 + 0.001 * ep)          # 衰减学习率
            a = np.random.randint(2)
            s2, r = step(s, a)
            Q[s, a] += alpha * (r + GAMMA * np.max(Q[s2]) - Q[s, a])
            s = s2
            ep += 1
        if ep % 200 == 0:
            errs.append(np.max(np.abs(Q - Q_true)))
    return errs

errs = ql()
print('Q-learning 到真值的 max 误差(每 200 轮):')
print('  ', [f'{e:.3f}' for e in errs])
print('→ 无限探索 + 衰减 α → 误差单调下降逼近 0 (收敛性质实证)')


Q-learning 到真值的 max 误差(每 200 轮):
   ['0.000', '0.000', '0.000']
→ 无限探索 + 衰减 α → 误差单调下降逼近 0 (收敛性质实证)


## 26. 面试追问速答终版（🔴 03 篇收官 2/2）

9. **Q: 表格 Q 与深度 Q 的收敛区别？** A: 表格收敛到 Q*；深度 Q 只是函数近似，无保证。
10. **Q: 经验回放的优先采样与均匀采样的区别？** A: 均匀简单；PER 加权难样本，方差小但需重要性修正。
11. **Q: 期望 SARSA 与 Q-learning 的关系？** A: ε=0 时退化为 Q-learning；一般更平滑。
12. **Q: Double Q 为什么不需要目标网络？** A: 两表交替扮演"目标"；深度版才用 target 网络。
13. **Q: 探索-利用的最优平衡？** A: 理论上 Gittins index/贝叶斯最优；实践靠退火+经验。
14. **Q: 为什么不用纯随机策略？** A: 无方向性——引导式探索（softmax/UCB）更高效。
15. **Q: 老虎机算法的遗憾界？** A: UCB O(√(KT log T)) 最优；ε-greedy 线性遗憾。
16. **Q: RL 里"策略梯度"与"Q 学习"能结合吗？** A: 能——AC 就是结合（策略+价值）。

---

## 27. 自测清单（03 篇冲刺版）

- [ ] 20 秒说出探索方法全景 7 种
- [ ] 复现 Q-learning 收敛到 Q* 实验并解释三个条件
- [ ] 能答出 16 连问中的随机 5 题
- [ ] 表格经验回放与 DQN 回放的区别一句话讲清


## 28. 考前 3 分钟速背（🔴 Q/SARSA 篇）

```text
Q-learning: r+γ·maxQ(s')           off / 激进
SARSA     : r+γ·Q(s',a')            on  / 保守
期望SARSA : r+γ·ΣπQ(s',·)           平滑
Double Q  : 选A估B解耦              抗高估
ε 衰减    : 先探索后利用
悬崖结论  : SARSA 安全绕路, Q 走最短
off-policy 红利: 数据复用 → 回放池
```


## 29. 常见追问组合演练（🔴 面试对练）

- **A**: "为什么 Q-learning 是 off-policy？" → 目标用 max、与实际采取动作无关
- **B**: "off-policy 有什么好处？" → 数据复用（回放）、学任意策略（如最优）
- **C**: "SARSA 什么时候更合适？" → 安全敏感（无人机/机器人），想让策略"知道探索会摔"
- **D**: "Double Q 为什么能抗高估？" → 选择与估值解耦，噪声不在同一张表放大


## 30. 追踪实验：多 seed 稳定性报告（🟡 RL 实验规范）

RL 结果必须多 seed：同样参数跑 5 个种子，报告
- 中位数 + 四分位（不是单次均值！）
- 样本方差带曲线

> 面试展示项目时随口说"5 seeds 中位数"，直接和专业拉开差距。


In [14]:
# multi_seed.py —— 5 seed 稳定报告示范(Q-learning 悬崖)
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

def run_seed(seed, episodes=300):
    np.random.seed(seed)
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    alpha, gamma, eps = 0.5, 0.9, 0.1
    tail = []
    for _ in range(episodes):
        s = START
        total, steps = 0.0, 0
        while s != GOAL and steps < 200:
            a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
            s2, r = step(s, a)
            Q[s[0], s[1], a] += alpha * (r + gamma * np.max(Q[s2[0], s2[1]]) - Q[s[0], s[1], a])
            s = s2
            total += r
            steps += 1
        tail.append(total)
    return np.mean(tail[-100:])

results = [run_seed(s) for s in range(5)]
arr = np.array(results)
print(f'5 seeds 后100回合均值: {np.round(arr, 1)}')
print(f'中位数: {np.median(arr):.1f}, 四分位: [{np.percentile(arr, 25):.1f}, {np.percentile(arr, 75):.1f}]')
print(f'均值±方差: {arr.mean():.1f} ± {arr.std():.1f}')
print('→ 报告请用中位数+四分位, 不用单次均值(可能被离群 seed 带偏)')


5 seeds 后100回合均值: [-43.  -36.4 -47.6 -40.8 -37.8]


中位数: -40.8, 四分位: [-43.0, -37.8]
均值±方差: -41.1 ± 3.9
→ 报告请用中位数+四分位, 不用单次均值(可能被离群 seed 带偏)


## 31. 模块衔接（🔴 为什么下一篇是 DQN）

表格 Q 在 16 个格子很好用，但状态一多就爆炸（00 篇算过 $10^{17000}$）。把 $Q(s,a)$ 换成神经网络 $Q_\theta(s,a)$，配合回放/目标网络——就是 DQN。

> 记忆桥：Q-learning 的一行更新 = DQN 的一步梯度下降（目标相同：TD 误差最小化）。


In [15]:
# ql_to_dqn_bridge.py —— 代码桥: 表格 Q 更新 vs 神经 Q 更新的对照(同一 TD 目标)
import numpy as np

# 表格版(上一行)
Q_tab = np.zeros((3, 2))
def tab_update(s, a, s2, r, alpha=0.3, gamma=0.9):
    Q_tab[s, a] += alpha * (r + gamma * np.max(Q_tab[s2]) - Q_tab[s, a])

# 神经版(线性近似, 同一目标)
W = np.zeros((3, 2))
def nn_update(s, a, s2, r, alpha=0.3, gamma=0.9):
    q_online = W[s, a]
    y = r + gamma * np.max(W[s2])
    W[s, a] += alpha * (y - q_online)      # 相当于对参数做一步梯度

for i in range(10):
    tab_update(0, 0, 1, -1.0)
    nn_update(0, 0, 1, -1.0)
print('表格 Q[0,0] =', round(Q_tab[0, 0], 3))
print('神经 Q[0,0] =', round(W[0, 0], 3))
print('→ 两者更新的是同一个 TD 目标; 区别只在"查表" vs "参数"')


表格 Q[0,0] = -0.972
神经 Q[0,0] = -0.972
→ 两者更新的是同一个 TD 目标; 区别只在"查表" vs "参数"


## 32. 章末寄语（🟢）

表格时代三件套（Q/SARSA/期望SARSA）全部掌握后，你对 off/on-policy、探索、收敛条件的理解已经超过 90% 的候选人。下一篇把它们"放大"到深度学习。


## 33. 控制方法变种谱系（🔴 Q/SARSA/期望/Double/n步 更新目标推导）

### 一张表看全部变种（背）

| 变种 | TD 目标 | 特性 |
|---|---|---|
| Q-learning | $r+\gamma\max_{a'}Q(s',a')$ | off, 激进 |
| SARSA | $r+\gamma Q(s',a')$ | on, 保守 |
| 期望 SARSA | $r+\gamma\sum_{a'}\pi(a'|s')Q(s',a')$ | on, 低方差 |
| Double Q | $r+\gamma Q_B(s',\arg\max Q_A)$ | 抗高估 |
| n步 Q | $G_{t:t+n}+\gamma^n\max_a Q(s_{t+n},a)$ | 少偏差 |

### 统一推导视角（背）

**所有变种 = 同一骨架**：
$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[\, \underbrace{r+\gamma\,(\text{后继估计})}_{\text{TD 目标}} - Q(s,a)\,\right]$$

差异只在"后继估计"用 $\max$（off）还是用 $\pi$ 的期望（on）还是双表交叉（Double）。


## 34. 最大化偏差推导：E[max X] ≥ max E[X]（🔴 面试手推）

### 前提

估计噪声 $\epsilon \sim \mathcal{N}(0, \sigma^2)$ 叠加在真值上：$\hat Q_i = Q_i^* + \epsilon_i$

### 一步推导（背）

```text
① 对单个动作: E[ ε_i ] = 0          → 无偏
② 对 max:     E[ max_i (Q_i^* + ε_i) ]
              = Q_max^* + E[ max_i ε_i ]
③ 关键: E[ max ε ] > max E[ε] = 0   (凸函数, Jensen)
   → E[ max_i Q̂_i ] ≥ max_i Q_i^*    严格高估 ∎
```

> **口语例子**（背）：一个赌局期望 +0，但两个独立噪声里取 max 期望是正的——"从噪声里挑最大"本身就是偏置。"D"ouble DQN 用另一张表估值，让选择与估值解耦，专治这个。


In [16]:
# fig_qvssarsa.py —— 过程图①: 悬崖漫步 Q-learning vs SARSA 学习曲线
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c = state
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    if (nr, nc) in CLIFF:
        return START, -100.0
    if (nr, nc) == GOAL:
        return GOAL, 0.0
    return (nr, nc), -1.0

def learn(method, episodes=200, alpha=0.5, gamma=0.9, eps=0.1, seed=0):
    np.random.seed(seed)
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    ret = []
    for _ in range(episodes):
        s = START
        a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s[0], s[1]]))
        total, steps = 0.0, 0
        while s != GOAL and steps < 300:
            s2, r = step(s, a)
            if method == 'q':
                target = r + gamma * np.max(Q[s2[0], s2[1]])
            else:
                a2 = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s2[0], s2[1]]))
                target = r + gamma * Q[s2[0], s2[1], a2]
            Q[s[0], s[1], a] += alpha * (target - Q[s[0], s[1], a])
            s, a = s2, (a2 if method == 'sarsa' else (np.random.randint(4) if np.random.random() < eps
                                                     else int(np.argmax(Q[s2[0], s2[1]]))))
            total += r
            steps += 1
        ret.append(total)
    return np.array(ret)

fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=150)
window = np.ones(20) / 20
q = np.convolve(learn('q', seed=1), window, mode='valid')
s = np.convolve(learn('sarsa', seed=1), window, mode='valid')
ax.plot(q, lw=2, color='crimson', label='Q-learning（激进，早期掉崖）')
ax.plot(s, lw=2, color='royalblue', label='SARSA（保守，绕路安全）')
ax.set_xlabel('回合数（20 步滑动平均）'); ax.set_ylabel('累计奖励')
ax.set_title('悬崖漫步: on/off-policy 行为差异（Sutton 例 6.6 复现）')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join('autumn-recruit-algo', '07-强化学习', '教学', 'images')
if os.path.basename(os.getcwd()) == '教学':
    out = os.path.join(os.getcwd(), 'images')
os.makedirs(out, exist_ok=True)
fig.savefig(os.path.join(out, 'q_vs_sarsa_curves.png'), dpi=150)
print('已保存 images/q_vs_sarsa_curves.png')
plt.close(fig)
plt.show()


已保存 images/q_vs_sarsa_curves.png


C:\Users\24260\AppData\Local\Temp\ipykernel_12668\247978552.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# fig_doubleq_bias.py —— 过程图②: 最大化偏差数值演示(E[max]>max E)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

rng = np.random.RandomState(0)
true_q = np.array([0.0, 0.0])            # 两个动作真值都是 0
K = 2000
est_max = np.array([max(true_q + rng.randn(2) * 1.0) for _ in range(K)])
est_double = np.array([0.5 * (max(true_q + rng.randn(2) * 1.0)
                              + max(true_q + rng.randn(2) * 1.0)) for _ in range(K)])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.6, 4.2), dpi=150)
for ax, data, color, ttl in [
    (ax1, est_max, 'crimson', '单表 max（正偏）'),
    (ax2, est_double, 'royalblue', '双表交替（居中）')]:
    ax.hist(data, bins=40, color=color, alpha=0.7)
    ax.axvline(0, color='black', ls='--', lw=1, label='真值 0')
    ax.set_xlabel('Q 估计'); ax.set_ylabel('频数')
    ax.set_title(ttl, fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle('最大化偏差: max 会把噪声取偏(红); 双表解耦(蓝)居中', fontsize=11)
plt.tight_layout()
out = os.path.join('autumn-recruit-algo', '07-强化学习', '教学', 'images')
if os.path.basename(os.getcwd()) == '教学':
    out = os.path.join(os.getcwd(), 'images')
os.makedirs(out, exist_ok=True)
fig.savefig(os.path.join(out, 'double_q_bias.png'), dpi=150)
print('已保存 images/double_q_bias.png')
plt.close(fig)
plt.show()


已保存 images/double_q_bias.png


C:\Users\24260\AppData\Local\Temp\ipykernel_12668\2018428936.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📸 本章推导配图（教学/images/）

![悬崖漫步: Q-learning(激进) vs SARSA(保守) 学习曲线](images/q_vs_sarsa_curves.png)

> 悬崖漫步: Q-learning(激进) vs SARSA(保守) 学习曲线

![最大化偏差: E[max]≥maxE 的直方图证据（Double Q 解耦）](images/double_q_bias.png)

> 最大化偏差: E[max]≥maxE 的直方图证据（Double Q 解耦）
